In [ ]:
import pandas as pd
import numpy as np
import polars as pl

In [ ]:
soil_path = "../data/raw/soil_data.csv"
train_path = "../data/raw/train_timeseries.csv"
validation_path = "../data/raw/validation_timeseries.csv"
test_path = "../data/raw/test_timeseries.csv"

In [3]:
# Read data
soil = pl.read_csv(soil_path).drop_nulls()
train = pl.read_csv(train_path)
validation = pl.read_csv(validation_path)
test = pl.read_csv(test_path)

Merge timeseries data with soil data + simple feature engineering

In [4]:
dfs = [train, validation, test]

for i, df in enumerate(dfs):
    print(f'dataframe {i}: ', end='')
    df = (df
        .with_columns(
            pl.col("date").cast(pl.Date)
        )
        .with_columns(
                # Interpolate Score (only one value per week) and drop nulls
                pl.col("score").interpolate(),
                # Encode date into seasonal vector
                (2 * np.pi * pl.col("date").dt.ordinal_day() / 366).sin().alias("date_sin"),
                (2 * np.pi * pl.col("date").dt.ordinal_day() / 366).cos().alias("date_cos"),
        )
        .drop('date')
        .drop_nulls()
        .join(soil, how='left', on='fips')
    )
    dfs[i] = df
    print('done.')

dataframe 0: done.
dataframe 1: done.
dataframe 2: done.


Normalize data (sklearn StandardScaler seems to break with data of this size)

In [5]:
def normalize_df(df, mean, std, exclude_cols):
    exprs = []
    for col in df.columns:
        if col in exclude_cols:
            continue

        exprs.append(
            (pl.col(col) - mean[col]) / std[col]
        )
    return df.with_columns(exprs)

In [6]:
# dfs[0] is train
mean = dfs[0].mean()
std = dfs[0].std()
exclude_cols = ['fips', 'score', 'date_sin', 'date_cos']

In [7]:
for i, df in enumerate(dfs):
    print(f'dataframe {i}: ', end='')
    dfs[i] = normalize_df(df, mean, std, exclude_cols)
    print('done')

dataframe 0: done
dataframe 1: done
dataframe 2: done


Write to output files

In [12]:
dfs[0].write_csv("../data/processed/train_timeseries.csv")
dfs[1].write_csv("../data/processed/validation_timeseries.csv")
dfs[2].write_csv("../data/processed/test_timeseries.csv")